In [2]:
# Import required libraries
import os
from dotenv import load_dotenv
import pandas as pd
import chromadb

# Load environment variables
load_dotenv()

# Get ChromaDB collection name from environment variables
COLLECTION_NAME = os.getenv('COLLECTION_NAME')

# Load the dataset with embeddings
df = pd.read_excel('../data/merged_with_embeddings.xls')

def extract_channel_thumbnail(val):
    """
    Ensures we always store a single string URL.
    Handles dicts from YouTube API format or direct strings.
    """
    if isinstance(val, dict):
        # Use the 'high' URL if available, else 'medium', else 'default'
        for quality in ['high', 'medium', 'default']:
            if quality in val and 'url' in val[quality]:
                return val[quality]['url']
        return None
    elif isinstance(val, str):
        return val.strip() if val.strip() else None
    else:
        return None

df['channel_thumbnail'] = df['channel_thumbnail'].apply(extract_channel_thumbnail)

# Now initialize ChromaDB and insert data
client = chromadb.PersistentClient(path="./chromadb_data")
collection = client.get_or_create_collection(name="video_embeddings")

# Prepare
ids = df['id'].astype(str).tolist()
def parse_embedding(x):
    if isinstance(x, str):
        return eval(x)
    return x

embeddings = df['embedding'].apply(parse_embedding).tolist()

metadatas = df[['id', 'title', 'description', 'categoryId', 'duration',
                'channel_id', 'channel_title', 'channel_thumbnail',
                'channel_description', 'channel_subscriberCount',
                'likeCount', 'transcript']].to_dict(orient='records')

# Recreate or overwrite
client.delete_collection("video_embeddings")
collection = client.get_or_create_collection(name="video_embeddings")

collection.add(
    ids=ids,
    embeddings=embeddings,
    metadatas=metadatas
)

print("✅ Embeddings + cleaned channel thumbnails stored in ChromaDB!")

✅ Embeddings + cleaned channel thumbnails stored in ChromaDB!
